# Adapter as Intermediary

**Use case:** isotrieve works as a **transformation layer** that sits between your embedding call and the vector store — like a shim or proxy. It's not a database; it's middleware that changes the coordinate system of vectors.

**When you'd reach for this:** You want to use a new embedding model with an existing vector store, and you need a thin translation layer that works across frameworks (LangChain, LlamaIndex, raw API calls).

**What you need installed:** `isotrieve`, `numpy`, `langchain-core`, `llama-index-core`.

**Estimated runtime:** ~3 minutes.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krish1925/AECP/blob/main/isotrieve-python/notebooks/04_adapter_as_intermediary.ipynb)

In [ ]:
!pip install -q isotrieve numpy scikit-learn langchain-core llama-index-core

In [ ]:
import numpy as np
from isotrieve import RidgeMapping

print("imports OK")

## 1. The mental model: isotrieve as coordinate transform

Think of embedding models as different "coordinate systems" for representing text semantics. isotrieve learns a change-of-basis between them:

```
raw text
  |
  v
new embedder (768-dim)
  |
  v
[isotrieve inverse mapping]  <-- THIS IS THE INTERMEDIARY
  |
  v
old-space vector (384-dim)
  |
  v
vector store (still has old embeddings)
```

The vector store doesn't change. The embedder doesn't change. isotrieve sits in between and translates.

In [ ]:
from IPython.display import display, Markdown

diagram = """
```mermaid
flowchart LR
    A["Raw Text"] --> B["New Embedder\n(768-dim)"]
    B --> C{"isotrieve\nInverse Map"}
    C --> D["Old-Space Vector\n(384-dim)"]
    D --> E["Vector Store\n(unchanged)"]
    
    F["New Query"] --> B
    B --> C
    C --> G["Search Results"]
```
"""
display(Markdown(diagram))

## 2. Fit the mapping

In [ ]:
rng = np.random.default_rng(42)

D_OLD = 384
D_NEW = 768
LATENT = 64
N_CAL = 1500

latent = rng.normal(size=(N_CAL, LATENT))
W_old = rng.normal(size=(LATENT, D_OLD)) / np.sqrt(LATENT)
W_new = rng.normal(size=(LATENT, D_NEW)) / np.sqrt(LATENT)

X_cal = (latent @ W_old)
X_cal = X_cal / np.linalg.norm(X_cal, axis=1, keepdims=True)
Y_cal = (latent @ W_new)
Y_cal = Y_cal / np.linalg.norm(Y_cal, axis=1, keepdims=True)

mapping = RidgeMapping(alpha="auto", seed=0)
mapping.fit(X_cal, Y_cal)
mapping.save("intermediary_mapping.isotrieve")

print(f"Mapping fitted and saved: {mapping.d_src} -> {mapping.d_tgt}")

## 3. The same mapping, used three ways

The `.isotrieve` file is framework-agnostic. The same mapping object works with:
- Raw numpy (no framework)
- LangChain's `Embeddings` interface
- LlamaIndex's `BaseEmbedding` interface

### 3a. Raw numpy (no framework)

In [ ]:
# Simulate a new-model embedding
new_embedding = rng.normal(size=D_NEW)
new_embedding = new_embedding / np.linalg.norm(new_embedding)

# Apply the intermediary: new-space -> old-space
old_space_vector = mapping.inverse_transform(new_embedding.reshape(1, -1)).ravel()

print(f"New-space vector:  shape={new_embedding.shape}, norm={np.linalg.norm(new_embedding):.4f}")
print(f"Old-space vector:  shape={old_space_vector.shape}, norm={np.linalg.norm(old_space_vector):.4f}")

### 3b. LangChain adapter

In [ ]:
from isotrieve.adapters.langchain import IsotrieveEmbeddings

class FakeNewModelEmbedder:
    """Simulates a LangChain Embeddings implementation using the new model."""
    
    def embed_documents(self, texts):
        rng2 = np.random.default_rng(1)
        vecs = rng2.normal(size=(len(texts), D_NEW))
        return (vecs / np.linalg.norm(vecs, axis=1, keepdims=True)).tolist()
    
    def embed_query(self, text):
        return self.embed_documents([text])[0]

base = FakeNewModelEmbedder()
isotrieve_embeddings = IsotrieveEmbeddings(mapping, base)

# Embed documents — automatically mapped to old space
docs = ["Hello world", "Machine learning is great", "Vector databases are useful"]
embedded = isotrieve_embeddings.embed_documents(docs)

print(f"LangChain adapter: {len(docs)} docs embedded")
print(f"Output dim: {len(embedded[0])} (old model space)")
print(f"Expected dim: {D_OLD}")
assert len(embedded[0]) == D_OLD, "Dimension mismatch!"
print("OK — vectors are in old-space, ready for the existing store")

### 3c. LlamaIndex adapter

In [ ]:
from isotrieve.wrappers.llamaindex import IsotrieveEmbedding

class FakeLlamaIndexEmbedder:
    """Simulates a LlamaIndex BaseEmbedding."""
    
    def _get_query_embedding(self, query):
        rng2 = np.random.default_rng(2)
        vec = rng2.normal(size=D_NEW)
        return (vec / np.linalg.norm(vec)).tolist()
    
    async def _aget_query_embedding(self, query):
        return self._get_query_embedding(query)
    
    def _get_text_embedding(self, text):
        return self._get_query_embedding(text)
    
    def _get_text_embeddings(self, texts):
        return [self._get_text_embedding(t) for t in texts]
    
    async def _aget_text_embedding(self, text):
        return self._get_text_embedding(text)
    
    async def _aget_text_embeddings(self, texts):
        return self._get_text_embeddings(texts)

llama_base = FakeLlamaIndexEmbedder()
isotrieve_llama = IsotrieveEmbedding(
    new_model_embedder=llama_base,
    transform_artifact_path="intermediary_mapping.isotrieve",
)

# Query embedding — mapped to old space
query_vec = isotrieve_llama._get_query_embedding("What is machine learning?")

print(f"LlamaIndex adapter: query mapped to {len(query_vec)} dims (old model space)")
assert len(query_vec) == D_OLD
print("OK — same mapping, different framework")

## 4. Why this matters

The mapping is a **single .isotrieve file** that works across:
- Raw Python / numpy
- LangChain
- LlamaIndex
- OpenAI SDK (via `IsotrieveOpenAI` shim)
- ChromaDB (via `IsotrieveChromaFunction`)
- Qdrant (via `QdrantAdapter`)

You fit it once, deploy it everywhere. The mapping is embedder-agnostic middleware — it doesn't know or care what framework you're using.

In [ ]:
from isotrieve.mapping.base import read_isotrieve_header

header = read_isotrieve_header("intermediary_mapping.isotrieve")
print(f"Mapping type: {header['mapping_type']}")
print(f"Dimensions: {header['d_src']} -> {header['d_tgt']}")
print(f"Has inverse: {header['has_inverse']}")
print(f"File size: {__import__('os').path.getsize('intermediary_mapping.isotrieve') / 1024:.1f} KB")
print(f"\nThis single file is all you need to deploy the intermediary.")

## Try it yourself

Try fitting a `ProcrustesDiagMapping` instead of `RidgeMapping` and compare the holdout metrics. Which preserves pairwise geometry better?

```python
from isotrieve.mapping.linear import ProcrustesDiagMapping
mapping_diag = ProcrustesDiagMapping(seed=0)
mapping_diag.fit(X_cal[:, :D_OLD], Y_cal[:, :D_OLD])  # same dims required
```